# K - EN YAKIN KOMŞU
# (K - NEAREST NEIGHBORS)

Gözlemlerin birbirine olan benzerlikleri üzerinden tamin yapılır. akılda kalması açısından bana arkadaşını söyle sana kim olduğunu söyleyeyim der. bir gözlem biriminin kendine en yakın olan diğer gözlem birimleri hesaplanır ve bu diğer kendisine en yakın k adet gözlem biriminin bağımlı değişkeni neyse bunun için ilgili gözlem birimi için tahminde bulunulmuş olur. 

y     x1    x2
100   56    241
120   85    250
150   25    233
...   ...   ...
140   56    231

y bağımlı diğerleri bağımsız değişkenlerim, bağımlı değişkenim sayısal olduğu için bunun bir regresyon problemi olduğunu biliyorum. bu noktada bir tahmin etme işlemi gerçekleştirmek istiyorum

elimde şöyle bir gözlem birimi olsun
x1 = 50, x2 = 230 bu bağımsız değişken değerlerine sahip gözlem biriminin bağımlı değişkeni nedir? y'nin tahini değeri nedir?

öklid ya da benzeri bir uzaklık yöntemi ile bu gözlem biriminin her bir gözlem birimiyle arasındaki uzaklığı hesaplayıp, bu uzaklığı hesapladıktan sonra örn kendisine en yakın k tane gözlem biriminin bağımlı değişkeninin ortalamasını almamız gerekmektedir.

kökiçinde(ktoplamsembolü (x1 - y1)karesi)

bu işlem için bir nokta belirlenir ve bu nokta için en yakın uzaklıklar belirlenir, gözlem değerleri ile gözlem birimi kıyaslanır 
örn
(50-56)karesi + (230 - 241)karesi = 

karekökü(36 + 121) 157

daha sonra bunu tüm gözlem birimleri için yaparım, en küçük olan belirlediğimiz noktaya en yakın olacaktır. dolayısıyla noktama en yakın 5 noktanın bağımlı değişkeninin ortalamasını hesapladığımda noktam için tahmin sonucumu ortaya çıkarmış olurum.


knn hem regresyon hem sınıflandırma yöntemleri için kullanabildiğim bir yöntemdir. bunu sınıflandırma için yapsam ne yapacaktım, en yakın 5 gözlemin ortalaması değil de, en çok tekrar eden sınıfını seçiyor olacaktık. frekansı en sık gözlenen sınıf tahmin edilen sınıf olur.

# KEŞİFÇİ VERİ ANALİZİ (EDA)

kullanacak olduğumuz veri seti diabest olacak

1. Exploratory Data Analysis (keşifçi veri analizi) (veriyi tanımaya çalışırız)
2. Data Preprocessing & Feature Engineering  (veri önişleme ve özellik müh, eksikliklerini aykırılıklarını gereken düzeltme işlemlerini gerçekleştirip düzeltme işlemelrini yaparız)
3. Modeling & Prediction (model ve tahminleme)
4. Hyperparameter Optimization (hiperparametre optimizasyonu)(knnin dışsal bir parametresi olduğu için bunu optimize etmeyi öğreneceğiz)
6. Final Model (final modeli)

In [1]:
import pandas as pd
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_excel("DİABETES.xlsx")

In [3]:
def correct_bmi_value(value):
    """
    BMI sütunundaki tarih formatındaki değerleri gün.ay şeklinde float'a çevirir.

    Args:
        value: Düzeltilecek değer.

    Returns:
        Düzeltilmiş float değer veya hata durumunda None.
    """

    try:
        # DateTime'e dönüştürme işlemini daha spesifik hale getirme
        date_value = pd.to_datetime(value, errors='coerce', format='%Y-%m-%d')  # Varsayılan tarih formatı
        if pd.notna(date_value):
            # Gün ve ay bilgisini birleştirerek float'a çevir
            return float(f"{date_value.day}.{date_value.month}")

    except (ValueError, TypeError) as e:
        # Spesifik hata türlerini yakalama
        print(f"Hata: {e} - Değer: {value}")
        return None  # Hatalı değerleri None olarak ayır

    # Eğer datetime'e dönüştürme başarısız olursa, zaten bir sayıysa dönüştürmeye çalış
    try:
        return float(value)
    except ValueError:
        print(f"Hata: Değer {value} ne sayısal ne de tarih formatında.")
        return None

# BMI sütununu düzelt
df['BMI'] = df['BMI'].apply(correct_bmi_value)

In [4]:
# BMI sütununu düzelt
df['DiabetesPedigreeFunction'] = df['DiabetesPedigreeFunction'].apply(correct_bmi_value)
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,1.100,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [5]:
df.info() #info attım

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [6]:
df.head() #head attık

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,1.100,33,1


In [7]:
df.shape #shapeine baktım

(768, 9)

In [8]:
df.describe().T #veri setimin dağılımsal istatistiklerine baktım

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.0
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.0
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.0
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.0
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.0
BMI,768.0,28.776693,12.128440,1.100,25.17500,30.9000,35.90000,67.1
DiabetesPedigreeFunction,768.0,0.455991,0.278901,0.078,0.24375,0.3725,0.62625,1.6
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.0
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.0


In [9]:
df["Outcome"].value_counts() #bağımlı değişkenimin dağılımına baktım

0    500
1    268
Name: Outcome, dtype: int64

# VERİ ÖN İŞLEME(DATA PRE-PROCESSING) & FEATURE ENGINEERING

In [10]:
#Ana odağım knn modelini öğrenmek olduğu için buraları hızlı geçiyorum

In [11]:
y = df["Outcome"]
X = df.drop(["Outcome"], axis=1)
# uzaklık temelli yöntemlerde ve gradient distance temelli yöntemlerde değişkenlerin 
#standart olması elde edilecek sonuçların daha hızlı ya da daha doğru olmasını sağlayacaktır
#bu sebeper elimizdeki bağımsız değişkenleri standartlaştırma işlemine sokacağız,
X_scaled = StandardScaler().fit_transform(X) # standardScaler ile bağımsız değişkenlerimizi standartlaştırıyoruz

In [12]:
X_scaled #şu an bir np arrayi dönüyor ancak bu np arrayi istediğimiz bilgileri 
# şu anda barındırmıyor. yani sütun isimleri yok bunu eklemem gerekmektedir.

array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.39794488,
         0.6135538 ,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.17958709,
        -0.37669078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -0.45185216,
         0.77500672, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.21258892,
        -0.7570021 , -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ...,  0.1091789 ,
        -0.38386646,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ...,  0.13393027,
        -0.50585312, -0.87137393]])

In [13]:
X = pd.DataFrame(X_scaled, columns=X.columns)
#ölçeklendirilmiş X'leri alıyoruz bunu bir dfye çeviriyoruz ve bunların ilk halini
#alıyoruz oradaki sütun isimlerini giriyoruz.

In [14]:
X # burada standartlaştırılmış değerlerim gelmiş oldu.

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,0.639947,0.848324,0.149641,0.907270,-0.692891,0.397945,0.613554,1.425995
1,-0.844885,-1.123396,-0.160546,0.530902,-0.692891,-0.179587,-0.376691,-0.190672
2,1.233880,1.943724,-0.263941,-1.288212,-0.692891,-0.451852,0.775007,-0.105584
3,-0.844885,-0.998208,-0.160546,0.154533,0.123302,-0.055830,-1.036854,-1.041549
4,-1.141852,0.504055,-1.504687,0.907270,0.765836,1.181738,2.310603,-0.020496
...,...,...,...,...,...,...,...,...
763,1.827813,-0.622642,0.356432,1.722735,0.870031,0.340192,-1.022502,2.532136
764,-0.547919,0.034598,0.046245,0.405445,-0.692891,0.661959,-0.416157,-0.531023
765,0.342981,0.003301,0.149641,0.154533,0.279594,-0.212589,-0.757002,-0.275760
766,-0.844885,0.159787,-0.470732,-1.288212,-0.692891,0.109179,-0.383866,1.170732


# MODELLEME

Bu bölümde knn modelimizi oluşturacağız ve predictionları bulacağız

In [15]:
knn_model = KNeighborsClassifier().fit(X, y) #fit diyerek bağımlı ve bağımsız değişkenlerimi giriyorum
#knn modelimi çağırıyorum ve bu metod der ki benim bir komşuluk parametrem var dışarıdan gelen
# bunu belirtmen lazım, ama bunu şimdilik bilmediğim için bunu pas geçiyorum

In [16]:
# şimdi modelimin eğitim işini tamamlamış oldum. bağımlı ve bağımsız 
# değişkenlerim arasındaki bağlantıyı öğrenmiş oldum.

In [17]:
#şimdi tahmin yapmak istiyorum ancak, şimdilik rastgele bir user seçerek yapacağım
random_user = X.sample(1, random_state=45) # şimdi rastgele bir tahmin yapacağım için random_stateti rastgele seçtim
random_user

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
195,0.342981,1.161295,0.770014,1.283638,1.130518,0.876471,-0.218826,-0.360847


In [18]:
#şimdi kurmuş olduğum modele sorayım bakayım bu kullanıcının diyabet 
#olup olmadığını diyorum

knn_model.predict(random_user) #predict tahmin et der ve hangi özellikleri sormamızı istediğimmizi bize sorar
#random userı buraya soruyorum ve gönderiyorum

#bir tane kullanıcı olarak yolladığımda tahmin sınıfı geldi

array([1], dtype=int64)

In [19]:
#model fit etmek ayrı bir süreçtir, fit edilmiş modeli kullanarak tahmin 
#etmek ayrı bir süreçtir
# şimdi bütün model için tahminlerimi ypacağım ve bu tahminlerimi model doğrulama 
#ile kontrol edeceğim

# MODEL BAŞARI DEĞERLENDİRME

In [20]:
#Bu bölümde model başarısı değerlendirme işlemlerini yapacağız 
#tek bir gözlem birimi için bu tahminlemeyi yapmıştık şimdi tamamı için yapacağız
y_pred = knn_model.predict(X) #tüm gözlem birimleri için bu tahminlemeleri yapıp bunları saklamamız lazım
#bunu confision matrixi hesaplamak için kullanacağız. bunun üzerinden accuracy ve f1 gibi metrikleri hesaplıyor
# olacağız ve auc'yi hesaplama ihtiyacımız vardı, bunu da olasılık değerleri üzerinden yapabiliyorduk. roc eğrsinin altında kalan alanı 
#ifade eden auc'yi hesaplamak istiyoruz. bunu hesaplamak için roc eğrisi üzerinden gidiyorduk. classification thresholdları değiştirerek
#true pozitif rateler ile false pozitif ratelere göre roc eğrilerini oluşturuyoruz. ve bu alanın altında kalan alanlar aucyi oluşturuyor. 
#bu sefer tahmin edilen değerler değil de 1 sınıfına ait olma olasılıkları lazım bana auc ve rocu hesaplayabilmek için
#bunun için knn modeli çağıracağım ve predict_probayı kullanacağım ve bağımsız değişkenleri verdikten sonra 
#0. değil 1. indexteki değerleri getiriyorum. yani 1 sınıfına ait olma olasılıklarını getiriyorum.
#bu olasılıklar üzerinden roc_auc skorunu hesaplıyor olacağız. karmaşıklık matrisi için tahmin edilen değerler
# elimizde, artık classification report metodunu getirerek hesaplama işlemini gerçekleştiriyorum.

y_prob = knn_model.predict_proba(X)[:, 1]

print(classification_report(y, y_pred))

#hatırlayalım burayı 1 ve 0 sınıfına göre hesaplama işlemleri yapıyordu
#auc değerim 0.81 geldi bu ne demekti, başarılı sınıflandırma oranı demekti
#1e 1 dediğimiz ve 0a 0 dediğimiz durumlar bölü tüm durumlar demektir.
# dolayısıyla %81 doğru sınıflandırıyoruz demektir. bu ne demek her 100 kişiden
# 81ine diyabet ya da diyabet değil dediğimde doğru şekilde sınıflandırıyorum, % 19'da 
# yanlış sınıflandırıyorum demek oluyor. ancak dengesiz veri problemi varsa accu her zaman 
#doğru bilgi vermeyebilir demiştik. bu durumda başka metriklere de bakmak lazım demiştik
#bu metriklerden biri precision diğeri recall diğeri ikisinin harmonik ortalaması olan 
#f1 scoruydu. bunlar nasıl yorumlanıyordu hatırlayalım. precision neydi, 1 olarak tahmin ettiklerimin
# başarısıydı, recall da gerçekte 1 olanlar 1 olarak tahmin etme başarımızı ifade ediyordu
#f1 de harmonik ortalamalarıdır. doalyısıyla accuracy %81 ancak gerçekte 1 olan sınıfı tahmin
# etme başarımız o kadar da yüksek değill 0.66dır. 1 olması gerekenlerin tahmin başarısı biraz daha
# yüksek.f1 skorum 0.71 yani %70in üstündeyim ancak bu bbaşarıyı geliştirmem gerekmektedir.
#fikir vermesi açısından aritmatik ve ağırlıklı ortalamalar da değerlendirilebilir. 
#şu anda buradan değerlendireceğim 0.71 ve 0.81 değerlerim olacak.

              precision    recall  f1-score   support

           0       0.83      0.89      0.86       500
           1       0.77      0.66      0.71       268

    accuracy                           0.81       768
   macro avg       0.80      0.77      0.78       768
weighted avg       0.81      0.81      0.81       768



In [21]:
# bir diğer skorum roc_auc skorumdur

roc_auc_score(y, y_prob)
#auc skorum da %89 çıktı oldukça yüksek oldukça iyi bir değer. yani farklı threshold değerleri
#belirlenmiş olsa durumunu da göz önünde bulundurarak yani dengesiz veri durumunu göz önünde bulundurarak
#f1 skoru yanımıza aldık, veri dengeli olsaydı accuracyi yanımıza alacaktık ancak yine de yanımıza aldık, 
# dolayısıyla 3 tane metrik üzerinden genel başarımızı görüyoruz. farklı sınıflandırma aralıklarına göre başarım 
#0.88 , veri dengesiz gibi kabul edilirse başarım, 0.71 şeklinde kabul ederim. kayde değer bir nokta.
#modeli kurduğumuz veri de test ettik dikkat etmemiz gerek. yani bütün veriyle bir model kurduk, kurmuş olduğumuz 
# modelin başarısını yine modeli kurarken kullandığımız veriyle test ettik. aslında yapılması gereken şey nedir
#modelin görmediği verideki performansıyla değerlendirmektir. bunun için iki yol öğrendik, birincisi holdout yöntemi
# veri setini ikiye bölerek sınama yaklaşımı 802 20 gibi bir bölümüyle test bir bölümüyle eğitim yapmaktır, 
#ikincisi cross validation yöntemi aslında holdout yöntemindeki bazı ortaya çıkarabilecek dezavantajları ortadan kaldırmak
# için önerilmiştir. dolayısıyla cross kullanacağız. cross kullanarak örn 5 katlı çarpraz doğrulama yaparak hatamızı
# değerlendirelim bakalım bu hatalarla örtüşüyor mu? 

# not alalım: accu = 0.81
#f1 = 0.71

0.8857910447761195

In [22]:
#5katlı veya 10 katlı çarpaz doğrulama ile modelimi doğrulamak istiyorum

cv_results = cross_validate(knn_model, X, y, cv=5, scoring=["accuracy", "f1", "roc_auc"]) # model nesnemi ekliyorum, bağımsız ve bağımlı değişkenleri verdim
#kaç katlı istediğini söyle, bir de bunları yaparken kullanmak istediğin metrikleri ver, dikkat 
#crossvalscore diye bir metod var cross vale göre farkı birden fazla metriğe göre değerlendirme yapabilmesidir.
cv_results

#fit süreleri ve score time tahmin ve eğitim sürelerini barındırmaktadır bunlar kafa karışıklığı yaratmasın
#accu, f1 ve roc_auc değerini 5 katlı yaptığım için veri setini 5e böldü, 4üyle model kurdu biriyle test etti
#ilk yaptığı işlemde elimde bir tane hata var (roc_aucta ilk seçenek), daha sonra diğe dördüyle model kurup biriyle test etti 
#bunu hepsi için yaptı. dolayısıyla bunların ortalamasını alırsam 5 katlı çapraz doğrulamanın ortalamasını almış olacağım
#aşağıda bu işlemleri gerçekleştirelim.

{'fit_time': array([0.00097847, 0.00557995, 0.00617814, 0.00200963, 0.00100827]),
 'score_time': array([0.00595593, 0.00201368, 0.00566435, 0.00464749, 0.00509048]),
 'test_accuracy': array([0.73376623, 0.7012987 , 0.74675325, 0.77777778, 0.74509804]),
 'test_f1': array([0.57731959, 0.54901961, 0.56179775, 0.65306122, 0.59793814]),
 'test_roc_auc': array([0.78546296, 0.74305556, 0.75888889, 0.80896226, 0.79415094])}

In [23]:
print("accuracy: ", cv_results["test_accuracy"].mean())
print("f1: ", cv_results["test_f1"].mean())
print("roc_auc: ", cv_results["test_roc_auc"].mean())

accuracy:  0.7409387997623291
f1:  0.5878272634201369
roc_auc:  0.7781041229909155


önceki skorlarımla değerlendiriyorum, oysa f1 skorum başarılı gibiydi aslında:

accu = 0.81  , auc değerim 0.74'e düştü
f1 = 0.71 , f1 skorum 58e düştü, 
roc_auc = 0.88 , 0.77'ye düştü

burdan ne anlamalıyım? 
modeli kurduğumuz veriyi modelin performansını değerlendirmek için kurduğumuzda aslında ortaya bir miktar yanlılık çıkıyor. bu yanlılık sonuçları doğru değerlendirmemi engelliyor. normalde burada dengesiz bir veri problemi var mı yok mu gibi bir soru işareti oluştuğunda f1 skorunu 71 gördüğümde bu beni problem var düşüncesinden uzaklaştırabilir. tamam bir miktar dengesizlik var gibi ama çok ciddi değil sanırım göz ardı edilebilir gibi bir yorum yaptırabilir bize, bununla beraber auc değeri çok yüksek bu da diğer metriklerin etkisini gözümüzden düşürebilir, iyi yoldayız gibi düşündürebilir. dolayısıyla burada veri setini bölerek ayrı parçalarında model kurarak test verisini de görmediği veri de test ederek daha güvenilir hale getiriyorum. sonuçlarımın ne kadar doğru olduğunu test etme ihtiyacımdan doğan işlemlerdir bunlar.


peki bu başarı sonuçları nasıl artırılabilir?

1. veri boyutu artırılabilir.
2. veri ön işleme işlemleri detaylandırılabilir
3. özellik mühle yeni değişkenler türetilebilir
4. ilgili algoritma iin optimizasyonlar yapılabilir

knn yönteminin dışsal parametresi vardır, komşuluk sayısı hiperparametresi vardır, bu parametre değiştirilebilirdir.

In [24]:
# parametre ayarı yapacağımm

knn_model.get_params() #parametrelerini getir bakalım dedim.
#burada komşuluk sayısı 5 oalrak gözüküyor.

#parametere modellerin veri içerisinden öğrendiği ağırlıklardı, 
# ağırlıklar parametrelerin tahmincileridir. özetle parametre dediğimiz şey
# veri içerisinden öğrenilmektedir. hiperparametre ise kullanıcı tarafından 
# tanımlanması gereken dışsal ve veri seti içerisinden öğrenilemeyen parametrelerddir
#biz ne yapacağız,yine veriyi kullanrak kullanıcı olarak bir hiperparametre seti vereceğiz
#10 tane 20 tane, 30 tane, komşuluk sayısı hpsi vereceğiz, bunları tek tek deneyeceğiz ve 
#denemeler neticesinde hatamıza bakıp böylece normalde veriden öğrenilemeyen kullanıcı 
#tarafından verilmesi gereken bu hpleri bir set olarak gönderip al bakalım 6 ya bak, 10a bak
# 20ye bak... dene bunları, ondan sonra en düşük hata veren hp setini bana nihai olarak göster 
#bende ondan sonra tekrar final bir model kurayım bu hp değerinin ön tanımlı değerini değiştireyim
#ona göre birdaha hatama bakayım diyerek algoritmanın hiperparametresini optimize edeceğim.

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

# HİPERPARAMETRE OPTİMİZASYONU

In [25]:
# bu bölümde makine öğrenmesi için oldukça önemli bir bölümdür, derin öğrenme, gelişmiş yöntemlerin çoğunda
#kullanıcıların ayarlaması gereken birçok parametre vardır. dolayısıyla bundan sonra sürekli 
#karşımıza çıkacak olan bir konudur. doğusal regresyonda, lojistik regresyon bunları görmedik? evet ancak bunlar zaten
#pratikte nerdeyse hiç kullanılmaz zaten. iktisadi, ekonomik, istatistiksel bazı çalışmaları dışarıda tutarsak 
#genel hatları itibari ile makine öğrenmesi, veri bilimi problemlerinde, yani excel formatındaki verilerde ve 
#ses ve görüntülü verilerde de kullanılacak olan gelişmiş yöntemlerin hemen hemen hepsinde dışarıdan kullanıcının ayarlaması 
#gereken parametreler vardır. dolayısıyla hiperparametre optimizasyonu kapsamında bu ayarlarlamamız gereken dışsal parametrleri
#programatik şekilde en doğru nasıl ayarlayabileceğimizi öğreneceğiz.

In [26]:
#knn modelimiz vardı ve bu modelin ön tanımlı parametreleri vardı

knn_model = KNeighborsClassifier() #modelimi getirdim
knn_model.get_params() # params diyerek ön tanımlı parametrelere bakacağım

# bakıyorum komşuluk sayım 5, şimdi amacım bu komşuluk sayısını değiştirerek
#olması gereken en optimum komşuluk sayısının ne olacağını bulmak 

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [27]:
#bunun için şu şekilde bir parametre listesi oluşturuyoruz
knn_params = {"n_neighbors": range(2,50)} #bir sözlük açtım ve komşu sayımı buraya girdim
#ancak sözlük isimlendirmesinin paramstakinin aynısı olması gerekmektedir, iki nokta üst üste sözlük olduğu için
# ve 2den 50ye kadar sayılar oluşturdum. yani 2den 50ye kadar olan sayıları tek tek deniyor olacağız.

#gönderdim ve evet knn_params bir sözlük ve bu sözlüğün içerisinde bir parametre var, ifade ediliş tarzı parametrenin
#içindekiyle tamamen aynı olacak şekilde ve buna karşılık iki nokta üst üste diyerek şunları ara(2,50) şeklinde ifade 
#ettiğimiz bir yapıdır.

In [28]:
#bunları nasıl arayacağız, GridSearchCV diye bir metodumuz var.
knn_gs_best = GridSearchCV(knn_model, knn_params, cv=5, n_jobs=-1, verbose=1).fit(X,y) #tek bir parametre olduğu için bir olası kombinasyon seti yok 
#yani gidip şunu yapacağız, 3 komşuluk için knni kuracağız ve hatamıza bakacağız
# yine 4 komşuluk için knn kurup hatamıza bakacağız ve bunu 50ye kadar yapacağız.
#GridSearchCV metodu bize bunu programatik olarak kolayca sağlayacak. ama eğer kullanacak
#olduğumuz yöntemde birden fazla hiperparametre varsa bu durumda bunların bütün olası
#kombinasyonlarını seçip gidip deniyor olacak. bu metodu bu şekilde aklımda tutabilirim.

#GridSearchCV der ki bana modelini göster, modelimi verdim, sonra peki bu modele yönelik 
#hangi parametre setini denemek istiyorsun onu gönder, kaç katlı cv yapacaksın onu ver der,
#dikkat! hatamızı değerlendirmek için cv yapıyoruz bu ayrı bir konu, hiperparametre optimizasyonu
#için kullanıyoruz bu ayrı bir konu, burada odağımız şu, olası bir hiperparametre değeri ile bir model
#kur hatana bak diyeceğiz ya, o hataya bakma işlemini de 5 katlı yap diyorum. böylece hiperparametre seçimi
#için de aslında cv'yi kullanmış oluyorum. hatamızı 5 katlı değerlendiriyoruz. bunun dışında sonra
#n_jobs argümmanı var yaygınca kullanılan argümanalrdan biridir, -1 yapılması durumunda işlemcileri tam performans
#ile kullanır, bu sayede sonuçlara daha hızlı şekilde gidilebilir. sonra verbose argümanı var, gridsearchcv der ki
#yaptığım sonuçlarda denemeler yapacağım sen bir rapor bekler misin der evet rapor bekliyorum diyorum
#bu yüzden verbose'u 1 yapıyorum.daha sonra fit diyerek X ve y olan bağımlı ve bağımsız değişkenlerimi gönderiyorum
#isim şu yüzden best, bu arama neticesinde komşu sayısı en az hatayı verecek şekilde bulunmuş olacak,
#

Fitting 5 folds for each of 48 candidates, totalling 240 fits


In [29]:
#burada 48 tane aday varmış, yani 48 adet hiperparametre denenecekmiş ve her 
# birisi için cv yapılacağından dolayı toplam 240 adet fit etme işlemi varmış.
#işlem tamamlandı, noldu peki? GridSearchCV yöntemini kullanarak knn algoritması için
#optimum komşuluk sayısının ne olduğunu buldu bu yöntem, o zaman bunun içinden bunu çağıracağım
knn_gs_best.best_params_ #bakın komşuluk sayısı 25 olarak geldi. 
#ön tanımlı değerim 5ti, 25 komşuluk değeri ile bir final modeli kurarsam final
#modelinin başarısının daha iyi olmasını beklerim.
#çünkü artık modelin de dışarıdan ayarlanması gereken parametresini ayarlamış oldum.

{'n_neighbors': 25}

# FİNAL MODELİ

In [30]:
#bu bölümde final modelini kuracağız.şu anda elimdeki knn aşgoritmasının en iyi
#hangi hiperparametre ile çalışabileceğini tespit etik, ve bunu bu modele göstermemiz lazım
#daha öncesinde onun ön tanımlı değerini kullanmıştık. şimdi yapılması gereken madem bunun en iyi 
#değerini bulduk biz, bununla tekrardan model kurmamız lazım.
knn_final = knn_model.set_params(**knn_gs_best.best_params_).fit(X, y) #burada bulduğum final parametremi knn_modelime atacağım set_params
#diyorum, buraya vermem gereken bir tane parametre var, knn_best_paramsı vermem gerekmektedir. ancak
# dikkat elimdekey value şeklinde yani dic şeklinde veriler olduğunda bunları basitçe atayabilmek için
#2yıldız kullanmam gerekmektedir.basitçe bu şekilde bunları buraya ata demiş oldum, noramlde bunu
#elimle {'n_neighbors': 25} şeklinde yazmam gerekmektedir. bir tane iki tane olduğunda elimizle yazabiliriz ancak
#kullanacak olduğumuz yöntemlerde onlarca olacak, onları elimizle yazamayız.bu durumda GreadSearchCV'deki parametreleri
#2* ile seçerek set_paramsa atayabiliriz. şimdi bu parametrelerle model fit etmemiz gerekmektedir bu yüzden fit diyorum.
#yine bağımlı ve bağımsız ddeğişkenlerimi verdim. sonra gönderiyorum

In [31]:
#modeli kurduktan sonra bu modelin test hatasına bakmam gerekiyor.

cv_results = cross_validate(knn_final,
                            X,
                            y,
                            cv=5,
                            scoring=["accuracy", "f1", "roc_auc"]) 


print("accuracy: ", cv_results["test_accuracy"].mean())
print("f1: ", cv_results["test_f1"].mean())
print("roc_auc: ", cv_results["test_roc_auc"].mean())

#model test hatama bakacağım bunun için cv metodumu getirdim, sonra knn_final'i buraya
#gönderiyorum ve değişkenlerimi verdim, 5 katlı yap dedik, ve hata değerlendirme
#metriklerimi verdim.
#işlem tamamlandı
#75, 56, 80 çıktı

accuracy:  0.7552584670231729
f1:  0.5633578947400085
roc_auc:  0.807235150244584


accuracy:  0.7409387997623291
f1:  0.5878272634201369             #bundan önceki değerler#
roc_auc:  0.7781041229909155

en son bu şekildeydi, f1 skorum hariç diğerleri artmış, onun tüm metrikleri arttı, makine öğrenmesinde başarıyı nasıl artırabilirim sorusu olduğunda bunu gidermek için temelde 4 konu olduğunu düşünebiliriz. bunlardan birisi veri önişleme, birisi örnek boyutunu artırma, birisi özellik mühendisliği, 4.sü ise ilgili algoritma için optimizasyon, buradaki algoritma bir knn ve bunun bir komşuluk sayısı hiperparametresi var bu komşuluk sayısı hiperparametresini optimize ettik, olası komşuluk sayılarınna gittik denedik ve 25 komşuluk sayısı en düşük hata oranını veriyor, doalyısıyla 25i seçtik ve algoritmayı optimize ettik ve başarımızı yükselttik.

In [32]:
# akılda kalması açısından kapanışı şu şekilde yapabiliriz,
#elimde madem final modeli var, bir kişi üzerinde deneyeceğim

random_user = X.sample(1) # random stati kaldırdım
random_user #hastamı seçtim ve final modelimi değerlendireceğim

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
186,1.23388,1.88113,-0.05715,0.969998,3.605143,0.109179,0.5705,2.276873


In [33]:
knn_final.predict(random_user) #predict dedi ki tamam kullanıcıyı yolla 

array([1], dtype=int64)

In [34]:
#sonuçta diabet olabileceği çıktı.
# başka bir kullanıcı seçip deneyeceğim

random_user = X.sample(1) # random stati kaldırdım
random_user

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
344,1.23388,-0.810425,0.149641,-1.288212,-0.692891,0.661959,0.10408,2.02161


In [35]:
knn_final.predict(random_user) #diabet çıkmadı

array([0], dtype=int64)

şunu öğrenmiş olmam lazım, biz veriye eriştik, buradaki patterni öğrendik modelleme sonrasında başarımmızı değerlendirdik veartık elimizde bir model var, bu veri seti içerisindeki ilişkinin özütü , bu modeli kullanarak tahminlerde bulunabiliriz. ve kullandık biz de sonuçta tahminler yaptık.